# Auditoria de qualidade do bronze: financeiro e obra

Notebook de auditoria da **Fictoria Casa & Interiores** (empresa fictícia, dados 100% sintéticos). Lê o **bronze** (estado corrente por chave) com DuckDB e evidencia, achado a achado, a sujeira que a **silver** precisa tratar. Cada seção termina numa **Nota Técnica** (Observado · Por que importa · Ação) escrita a partir do que a célula mostrou; o conjunto das ações é o [catálogo de achados](../docs/08_catalogo_achados_silver.md), o contrato da silver.

> Reprodutível: `uv run notebooks-qualidade` reconstrói e reexecuta este notebook a partir de `src/interiores_fictoria/qualidade/achados.py`.

In [1]:
import duckdb, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from interiores_fictoria import duck

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
con, lake = duck.conectar()
print("lake:", lake.raiz)
print("cargas no bronze:", len(lake.listar("bronze", "_controle", "cargas")))


lake: /home/neviah/projetos/interiores-fictoria-bigdata/data/lake
cargas no bronze: 5


## 1. ACH-12 · Comissão apurada com percentual diferente do orçamento

In [2]:
df = con.execute("""SELECT c.tipo, COUNT(*) AS comissoes,
                  SUM(CASE WHEN c.tipo = 'VENDEDOR' AND c.p_comissao <> o.p_comissao_vendedor THEN 1
                           WHEN c.tipo = 'PARCEIRO' AND c.p_comissao <> o.p_comissao_parceiro THEN 1
                           WHEN c.tipo = '3D' AND c.p_comissao <> o.p_comissao_3d THEN 1 ELSE 0 END)
                      AS divergentes
           FROM b_comissao c JOIN b_orcamento o ON o.id = c.orcamento_id GROUP BY 1 ORDER BY 2 DESC""").df()
display(df)
metrica = con.execute("""SELECT AVG(CASE WHEN c.tipo = 'VENDEDOR' AND c.p_comissao <> o.p_comissao_vendedor THEN 1.0
                           WHEN c.tipo = 'PARCEIRO' AND c.p_comissao <> o.p_comissao_parceiro THEN 1.0
                           WHEN c.tipo = '3D' AND c.p_comissao <> o.p_comissao_3d THEN 1.0 ELSE 0.0 END)
           FROM b_comissao c JOIN b_orcamento o ON o.id = c.orcamento_id WHERE c.tipo <> 'SUPERVISOR'""").fetchone()[0]
print("métrica ACH-12:", metrica)

,tipo,comissoes,divergentes
0,VENDEDOR,3347,33.0
1,3D,1875,16.0
2,SUPERVISOR,1218,0.0
3,PARCEIRO,1155,17.0


métrica ACH-12: 0.010349694213580053


**Nota Técnica ACH-12**

- **Observado:** taxa observada de 1,03% na base analisada (métrica: `ACH-12`).
- **Por que importa:** Comissão paga com percentual diferente do negociado é dinheiro saindo errado; a régua da diretoria ('quanto pagamos de comissão') precisa do apurado E do esperado.
- **Ação (regra da silver):** `p_comissao_esperado` e `fl_comissao_divergente` na silver; a gold carrega o valor apurado (o que foi pago) e a diferença contra o esperado como medida.

## 2. ACH-13 · Parcelas vencidas sem pagamento e pagamentos em atraso

In [3]:
df = con.execute("""SELECT YEAR(p.dt_vencimento) AS ano, COUNT(*) AS parcelas,
                  SUM(CASE WHEN NOT p.fl_pago AND p.dt_vencimento < DATE '2026-09-04' THEN 1 ELSE 0 END)
                      AS vencidas_sem_pagamento,
                  SUM(CASE WHEN pg.dt_pagamento > p.dt_vencimento + INTERVAL 5 DAY THEN 1 ELSE 0 END)
                      AS pagas_com_atraso
           FROM b_parcela p LEFT JOIN b_pagamento pg ON pg.parcela_id = p.id
           GROUP BY 1 ORDER BY 1""").df()
display(df)
metrica = con.execute("""SELECT AVG(CASE WHEN NOT fl_pago AND dt_vencimento < DATE '2026-09-04' THEN 1.0 ELSE 0.0 END)
           FROM b_parcela""").fetchone()[0]
print("métrica ACH-13:", metrica)

,ano,parcelas,vencidas_sem_pagamento,pagas_com_atraso
0,2021,460,14.0,42.0
1,2022,1276,49.0,112.0
2,2023,1954,69.0,188.0
3,2024,3353,106.0,318.0
4,2025,5464,178.0,552.0
5,2026,7058,249.0,429.0
6,2027,842,0.0,0.0


métrica ACH-13: 0.0325868574508747


**Nota Técnica ACH-13**

- **Observado:** taxa observada de 3,26% na base analisada (métrica: `ACH-13`).
- **Por que importa:** Inadimplência e atraso são a diferença entre venda e caixa; a comissão do vendedor só é liberada depois do primeiro pagamento, então atraso no cliente vira atraso na equipe.
- **Ação (regra da silver):** `fl_vencida_sem_pagamento` e `dias_atraso_pagamento` na silver; fato de parcelas na gold com as datas de vencimento e pagamento no calendário.

## 3. ACH-14 · Auditoria guarda valores como texto com vírgula

In [4]:
df = con.execute("""SELECT ds_campo, COUNT(*) AS alteracoes,
                  MIN(vl_novo) AS exemplo_menor, MAX(vl_novo) AS exemplo_maior
           FROM b_auditoria_orcamento WHERE ds_campo IN ('vl_total_liquido', 'vl_desconto')
           GROUP BY 1 ORDER BY 2 DESC""").df()
display(df)
metrica = con.execute("""SELECT COUNT(*) FROM b_auditoria_orcamento
           WHERE ds_campo IN ('vl_total_liquido', 'vl_desconto') AND vl_novo <> '(criação)'""").fetchone()[0]
print("métrica ACH-14:", metrica)

,ds_campo,alteracoes,exemplo_menor,exemplo_maior
0,vl_total_liquido,33217,(criação),"999.119,49"
1,vl_desconto,33217,(criação),"999,96"


métrica ACH-14: 35848


**Nota Técnica ACH-14**

- **Observado:** 35.848 ocorrências na base analisada (métrica: `ACH-14`).
- **Por que importa:** A auditoria é a única trilha de renegociação (quantas vezes o valor mudou até fechar), mas os valores estão em texto pt-BR: '1.234,56'. Sem conversão, nenhuma média sai.
- **Ação (regra da silver):** `vl_antigo_num`/`vl_novo_num` convertidos (remove pontos, troca vírgula por ponto) só para campos monetários; `qtd_revisoes` por orçamento = alterações de `vl_total_liquido`.

## 4. ACH-15 · Ganhos sem recebimento e comissão ainda não apurada

In [5]:
df = con.execute("""SELECT YEAR(o.dt_cadastro) AS ano, COUNT(*) AS ganhos,
                  SUM(CASE WHEN r.orcamento_id IS NULL THEN 1 ELSE 0 END) AS sem_recebimento,
                  SUM(CASE WHEN c.orcamento_id IS NULL THEN 1 ELSE 0 END) AS sem_comissao
           FROM b_orcamento o
           LEFT JOIN (SELECT DISTINCT orcamento_id FROM b_recebimento) r ON r.orcamento_id = o.id
           LEFT JOIN (SELECT DISTINCT orcamento_id FROM b_comissao) c ON c.orcamento_id = o.id
           WHERE o.tipo_orcamento_id = 1 AND o.dt_cancelou IS NULL AND o.fase_id = 6
           GROUP BY 1 ORDER BY 1""").df()
display(df)
metrica = con.execute("""SELECT AVG(CASE WHEN r.orcamento_id IS NULL THEN 1.0 ELSE 0.0 END)
            FROM b_orcamento o
            LEFT JOIN (SELECT DISTINCT orcamento_id FROM b_recebimento) r ON r.orcamento_id = o.id
            WHERE tipo_orcamento_id = 1 AND dt_cancelou IS NULL AND o.fase_id = 6""").fetchone()[0]
print("métrica ACH-15:", metrica)

,ano,ganhos,sem_recebimento,sem_comissao
0,2021,132,2.0,2.0
1,2022,254,4.0,4.0
2,2023,432,2.0,2.0
3,2024,639,8.0,8.0
4,2025,1079,17.0,17.0
5,2026,868,4.0,24.0


métrica ACH-15: 0.010869565217391304


**Nota Técnica ACH-15**

- **Observado:** taxa observada de 1,09% na base analisada (métrica: `ACH-15`).
- **Por que importa:** Ganho sem recebimento é venda de valor zero (achado 03) ou contrato ainda não emitido; comissão sem apuração no mês corrente é normal (apura no mês seguinte). Distinguir os dois evita alarme falso.
- **Ação (regra da silver):** Nenhum preenchimento: a gold trata venda sem recebimento como 'sem contrato' e comissão como 'a apurar' pelo calendário de competência.

## 5. ACH-16 · Medições não realizadas e instalações com problema

In [6]:
df = con.execute("""SELECT 'medições agendadas' AS item, COUNT(*) AS total,
                  SUM(CASE WHEN dt_realizada IS NULL THEN 1 ELSE 0 END) AS pendentes_ou_nao_realizadas
           FROM b_medicao
           UNION ALL
           SELECT 'instalações', COUNT(*), SUM(CASE WHEN fl_problema THEN 1 ELSE 0 END) FROM b_instalacao""").df()
display(df)
metrica = con.execute("""SELECT AVG(CASE WHEN fl_problema THEN 1.0 ELSE 0.0 END) FROM b_instalacao WHERE dt_fim IS NOT NULL""").fetchone()[0]
print("métrica ACH-16:", metrica)

,item,total,pendentes_ou_nao_realizadas
0,medições agendadas,9451,499.0
1,instalações,3222,472.0


métrica ACH-16: 0.1563949635520212


**Nota Técnica ACH-16**

- **Observado:** taxa observada de 15,64% na base analisada (métrica: `ACH-16`).
- **Por que importa:** Problema na instalação é o que gera correção e atraso no pós-venda; medição agendada e não realizada distorce o prazo até a proposta.
- **Ação (regra da silver):** Preservado como está; a gold traz instalação e medição como fatos de obra com as flags.

## Encerramento

Os achados acima entram no catálogo com id, regra e taxa observada. A silver implementa as regras e presta contas: para cada achado, quantas linhas foram afetadas, com o veredito que reprova a si mesmo quando a contagem não bate.